# Análise offline de runtimes e Prometheus

Este notebook lê **somente** o dataset consolidado já baixado do Pod. Ele não abre túnel, não chama runtime e não consulta Prometheus.

Antes de iniciar o Jupyter, aponte `OBSERVABILITY_DATASET_DIR` para o diretório que contém `dataset-manifest.json` e os CSVs consolidados. Métrica ausente permanece ausente; nunca é interpretada como zero.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
DATASET_DIR = Path(os.environ.get("OBSERVABILITY_DATASET_DIR", "../data/runtime-observability")).expanduser().resolve()
manifest_path = DATASET_DIR / "dataset-manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Dataset não encontrado em {DATASET_DIR}. Defina OBSERVABILITY_DATASET_DIR antes de abrir o Jupyter."
    )
dataset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

def read_table(name):
    path = DATASET_DIR / name
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()

inventory = read_table("run-inventory.csv")
complete_runs = read_table("complete-runs.csv")
comparison_groups = read_table("comparison-groups.csv")
requests_df = read_table("all-requests.csv")
events = read_table("request-events.csv")
phase_metrics = read_table("request-phase-metrics.csv")
telemetry_by_request = read_table("telemetry-by-request.csv")
prometheus_samples = read_table("prometheus-samples.csv")
summary = read_table("experiment-summary.csv")
coverage = read_table("metric-coverage.csv")

print(DATASET_DIR)
print(f"runs completos: {dataset_manifest['runs_complete']} | comparáveis: {dataset_manifest['runs_comparable']} | excluídos: {dataset_manifest['runs_excluded']}")

## 1. Integridade da campanha

Execuções parciais ou com artefato ausente, CSV malformado ou SHA-256 divergente aparecem no inventário, mas não entram nas comparações abaixo.

In [ ]:
display(inventory.groupby(["status", "included_in_comparison"], dropna=False).size().rename("runs").reset_index())
excluded = inventory.loc[~inventory["included_in_comparison"].astype(bool)]
if not excluded.empty:
    display(excluded[["run_id", "runtime", "model", "status", "integrity_errors", "exclusion_reasons"]])
display(comparison_groups[["comparison_id", "run_count", "runtime_count", "runtimes", "cross_runtime_ready", "comparison_identity_complete"]])

## 2. Grupo de comparação

`comparison_id` fixa SHA do modelo e tokenizer, contexto, saída, workload, configuração comparável e hardware. Runtime e versão do runtime são as variáveis comparadas e, por isso, não separam o grupo. O comando e ambiente efetivos, seus fingerprints, flags de cache com valores, estados de KV/prefix cache e residência do modelo aparecem como dimensões do tratamento. Se houver mais de um grupo, defina `OBSERVABILITY_COMPARISON_ID`; o notebook nunca os agrega silenciosamente.

In [ ]:
available_comparison_ids = sorted(comparison_groups["comparison_id"].dropna().astype(str).unique()) if "comparison_id" in comparison_groups else []
if not available_comparison_ids:
    raise ValueError("Nenhuma run íntegra e comparável. Consulte run-inventory.csv.")
COMPARISON_ID = os.environ.get("OBSERVABILITY_COMPARISON_ID") or available_comparison_ids[0]
if COMPARISON_ID not in available_comparison_ids:
    raise ValueError(f"comparison_id desconhecido: {COMPARISON_ID}. Opções: {available_comparison_ids}")
if len(available_comparison_ids) > 1 and not os.environ.get("OBSERVABILITY_COMPARISON_ID"):
    print("Há vários grupos; usando somente o primeiro. Defina OBSERVABILITY_COMPARISON_ID para trocar.")
print("comparison_id selecionado:", COMPARISON_ID)
display(comparison_groups.loc[comparison_groups["comparison_id"] == COMPARISON_ID])

def select_comparison(frame):
    if frame.empty or "comparison_id" not in frame:
        return frame.iloc[0:0].copy()
    return frame.loc[frame["comparison_id"].astype(str) == COMPARISON_ID].copy()

complete_runs = select_comparison(complete_runs)
requests_df = select_comparison(requests_df)
events = select_comparison(events)
phase_metrics = select_comparison(phase_metrics)
telemetry_by_request = select_comparison(telemetry_by_request)
prometheus_samples = select_comparison(prometheus_samples)
summary = select_comparison(summary)
coverage = select_comparison(coverage)
treatment_columns = [
    "run_id", "runtime", "runtime_version", "runtime_process_ownership",
    "runtime_process_argv_json", "runtime_process_argv_fingerprint",
    "runtime_environment_json", "runtime_environment_fingerprint",
    "effective_cache_flags_json", "effective_cache_argv_json",
    "effective_cache_environment_json", "prefix_cache_state",
    "kv_cache_state", "kv_cache_configuration_json",
    "model_residency", "compatibility_reason",
]
display(complete_runs[[column for column in treatment_columns if column in complete_runs]])

## 3. Resultado por cenário

A tabela separa os blocos e as fases do benchmark. Para a comparação principal, filtre `phase == 'measure'`; warm-up continua disponível como diagnóstico.

In [ ]:
formal = summary.loc[summary["phase"] == "measure"].copy() if "phase" in summary else summary.copy()
summary_columns = [
    "experiment_id", "runtime", "runtime_version", "model", "block", "scenario",
    "prefix_cache_state", "kv_cache_state", "model_residency",
    "successful_request_rows", "failed_request_rows",
    "ttft_ms_mean", "ttft_ms_p50", "ttft_ms_p95",
    "decode_tokens_s_mean", "decode_tokens_s_p50", "decode_tokens_s_p95",
    "e2e_seconds_mean",
]
display(formal[[column for column in summary_columns if column in formal]].sort_values([column for column in ["scenario", "runtime"] if column in formal]))

## 4. Distribuições por requisição

Média isolada esconde caudas. Estes boxplots usam cada requisição bem-sucedida como unidade estatística.

In [ ]:
def first_numeric(frame, candidates):
    for column, factor in candidates:
        if column in frame:
            values = pd.to_numeric(frame[column], errors="coerce") * factor
            if values.notna().any():
                return values
    return pd.Series(index=frame.index, dtype=float)

measured = requests_df.copy()
request_phase_column = "benchmark_phase" if "benchmark_phase" in measured else "phase"
if request_phase_column in measured:
    measured = measured.loc[measured[request_phase_column] == "measure"].copy()
if "status" in measured:
    measured = measured.loc[measured["status"].astype(str).str.lower().isin(["successful", "success", "ok"])].copy()
measured["TTFT (ms)"] = first_numeric(measured, [("time_to_first_token_ms", 1), ("ttft_ms", 1), ("time_to_first_token_seconds", 1000)])
measured["Decode (tokens/s)"] = first_numeric(measured, [("decode_tokens_s", 1), ("decode_tokens_per_second", 1), ("tokens_per_second", 1)])
measured["experiment"] = measured["runtime"].fillna("?") + " · " + measured["source_run_id"].fillna("?")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, metric in zip(axes, ["TTFT (ms)", "Decode (tokens/s)"]):
    valid = measured.dropna(subset=[metric])
    if valid.empty:
        axis.text(0.5, 0.5, "métrica ausente", ha="center", va="center")
        axis.set_axis_off()
        continue
    hue = "scenario" if "scenario" in valid and valid["scenario"].nunique() > 1 else None
    sns.boxplot(data=valid, x="experiment", y=metric, hue=hue, ax=axis)
    axis.set_title(metric + " por requisição")
    axis.tick_params(axis="x", rotation=20)
fig.tight_layout()

## 5. Cobertura das métricas

Branco significa **não coletado**, não uso zero. Verifique esta matriz antes de concluir que um runtime não usou GPU, VRAM, swap ou KV cache.

In [ ]:
coverage_plot = coverage.copy()
if not coverage_plot.empty:
    coverage_plot["experiment"] = coverage_plot["runtime"].fillna("?") + " · " + coverage_plot["run_id"].fillna("?")
    matrix = coverage_plot.pivot_table(index="metric", columns="experiment", values="available", aggfunc="max")
    plt.figure(figsize=(max(8, 2 + matrix.shape[1] * 2), max(6, matrix.shape[0] * 0.35)))
    sns.heatmap(matrix.astype(float), vmin=0, vmax=1, cmap=["#eeeeee", "#1f9d8a"], cbar=False, linewidths=0.5)
    plt.title("Cobertura: verde = há amostras; cinza = ausente")
    plt.xlabel("runtime · experimento")
    plt.ylabel("métrica")
    plt.tight_layout()
display(coverage.loc[~coverage["available"].astype(bool), ["run_id", "runtime", "metric", "missing_reason"]].head(30))

## 6. Recurso por prefill e decode

Escolha uma série disponível. A comparação por `observed_phase` relaciona recurso saturado com TTFT alto (prefill) ou tokens/s baixos (decode).

In [ ]:
available_phase_metrics = sorted(phase_metrics["metric"].dropna().unique()) if "metric" in phase_metrics else []
preferred = [
    "inference_gpu_utilization_ratio",
    "inference_gpu_memory_used_bytes",
    "inference_host_cpu_utilization_ratio",
]
METRIC_TO_PLOT = next((metric for metric in preferred if metric in available_phase_metrics), available_phase_metrics[0] if available_phase_metrics else None)
print("Métricas disponíveis:", available_phase_metrics)
print("Métrica selecionada:", METRIC_TO_PLOT)
if METRIC_TO_PLOT:
    selected = phase_metrics.loc[phase_metrics["metric"] == METRIC_TO_PLOT].copy()
    if "benchmark_phase" in selected:
        selected = selected.loc[selected["benchmark_phase"] == "measure"]
    selected["mean"] = pd.to_numeric(selected["mean"], errors="coerce")
    selected["experiment"] = selected["runtime"].fillna("?") + " · " + selected["source_run_id"].fillna("?")
    plt.figure(figsize=(11, 5))
    sns.barplot(data=selected, x="observed_phase", y="mean", hue="experiment", errorbar=("pi", 95))
    plt.title(METRIC_TO_PLOT + " por fase observada")
    plt.xlabel("fase observada")
    plt.ylabel(selected["unit"].dropna().iloc[0] if "unit" in selected and selected["unit"].notna().any() else "valor")
    plt.tight_layout()

## 7. Linha temporal de uma requisição

A célula usa as amostras brutas, quando disponíveis, e sobrepõe os eventos da mesma requisição. Troque `REQUEST_UID` e `TIMELINE_METRIC` para investigar outro caso.

In [ ]:
timeline_samples = telemetry_by_request if not telemetry_by_request.empty else prometheus_samples
REQUEST_UID = timeline_samples["request_uid"].dropna().iloc[0] if not timeline_samples.empty and "request_uid" in timeline_samples and timeline_samples["request_uid"].notna().any() else None
raw_metrics = sorted(timeline_samples["metric"].dropna().unique()) if "metric" in timeline_samples else []
TIMELINE_METRIC = next((metric for metric in preferred if metric in raw_metrics), raw_metrics[0] if raw_metrics else None)
if REQUEST_UID and TIMELINE_METRIC:
    trace = timeline_samples.loc[(timeline_samples["request_uid"] == REQUEST_UID) & (timeline_samples["metric"] == TIMELINE_METRIC)].copy()
    x_column = next((column for column in ["request_elapsed_s", "elapsed_seconds", "timestamp_s", "timestamp"] if column in trace), None)
    if x_column is None:
        raise ValueError("Telemetria sem coluna temporal reconhecida.")
    trace[x_column] = pd.to_numeric(trace[x_column], errors="coerce")
    if x_column in ["timestamp_s", "timestamp"]:
        trace[x_column] = trace[x_column] - trace[x_column].min()
    trace["value"] = pd.to_numeric(trace["value"], errors="coerce")
    fig, axis = plt.subplots(figsize=(13, 5))
    sns.lineplot(data=trace, x=x_column, y="value", marker="o", ax=axis)
    request_events = events.loc[events["request_uid"] == REQUEST_UID] if "request_uid" in events else pd.DataFrame()
    for event in request_events.itertuples(index=False):
        elapsed = getattr(event, "request_elapsed_s", getattr(event, "elapsed_seconds", None))
        if pd.notna(elapsed):
            axis.axvline(float(elapsed), linestyle="--", alpha=0.7)
            axis.text(float(elapsed), axis.get_ylim()[1], str(getattr(event, "event", "evento")), rotation=90, va="top")
    axis.set_title(f"{TIMELINE_METRIC} · request {REQUEST_UID}")
    axis.set_xlabel("segundos desde o início da janela")
    fig.tight_layout()
else:
    print("Amostras Prometheus brutas não estão disponíveis neste dataset. Use a tabela por fase.")

## Sequência de conclusão

1. Confirme que as runs comparadas estão `complete` e usam a mesma carga.
2. Compare TTFT e tokens/s pelas distribuições, não por uma requisição isolada.
3. Confirme a cobertura antes de interpretar ausência de GPU, VRAM ou KV.
4. Localize a saturação em prefill ou decode.
5. Mude uma variável, repita a mesma carga e compare o antes/depois.